In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from scipy.stats import gaussian_kde
from mpl_toolkits.axes_grid1 import make_axes_locatable
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
import pandas as pd


plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'Times New Roman'
plt.rcParams['font.size'] = 20  


with rasterio.open(r'F:/S5P/data/ChinaHigh/tif/CHAP_NO2_Y1K_2023_V_NO2.tif') as src1:
    data1 = src1.read(1, masked=True)

with rasterio.open(r'F:/S5P/data/S5P/unit_converted_data/2023_CHINA_NO2_1Km.tif') as src2:
    data2 = src2.read(1, masked=True)


valid_indices = ~data1.mask & ~data2.mask
x = data1[valid_indices]
y = data2[valid_indices]


x = pd.to_numeric(x, errors='coerce')
y = pd.to_numeric(y, errors='coerce')


valid_mask = np.isfinite(x) & np.isfinite(y)
x = x[valid_mask]
y = y[valid_mask]

if len(x) == 0 or len(y) == 0:
    raise ValueError("No valid data points available for analysis.")

max_points = 10000
total_points = len(x)
if total_points > max_points:
    np.random.seed(0)  
    indices = np.random.choice(total_points, size=max_points, replace=False)
    x = x[indices]
    y = y[indices]


scaler = MinMaxScaler()
x_normalized = scaler.fit_transform(x.reshape(-1, 1)).flatten()
y_normalized = scaler.fit_transform(y.reshape(-1, 1)).flatten()


xy = np.vstack([x_normalized, y_normalized])
z = gaussian_kde(xy)(xy)

idx = z.argsort()
x_sorted, y_sorted, z_sorted = x_normalized[idx], y_normalized[idx], z[idx]

z_normalized = (z_sorted - z_sorted.min()) / (z_sorted.max() - z_sorted.min())

m, b = np.polyfit(x_sorted, y_sorted, 1)

y_pred = m * x_sorted + b

rmse = np.sqrt(mean_squared_error(y_sorted, y_pred))

r = np.corrcoef(x_sorted, y_sorted)[0, 1]

equation = f'y = {m:.2f}x + {b:.2f}'

fig, ax = plt.subplots(figsize=(10, 10), dpi=100) 

scatter = ax.scatter(
    x_sorted, y_sorted,
    marker='o',
    c=z_normalized,
    edgecolors='none',
    s=10,
    cmap='Spectral_r'
)

ax.plot(x_sorted, y_pred, 'k-', label='fitted-line')


divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)
cbar = fig.colorbar(scatter, cax=cax, label='density')


cbar.ax.set_ylabel('density', fontsize=20)
cbar.ax.tick_params(labelsize=20)


ax.set_xlabel('CH/µg/m³', fontsize=20)
ax.set_ylabel('S5P/µg/m²', fontsize=20)


ax.tick_params(axis='both', which='major', labelsize=20)

textstr = f'RMSE = {rmse:.2f}\nr = {r:.2f}\n{equation}'
props = dict(boxstyle='round', facecolor='white', alpha=0.8)
ax.text(
    0.05, 0.95, textstr,
    transform=ax.transAxes,
    fontsize=20,
    verticalalignment='top',
    horizontalalignment='left',
    bbox=props
)

ax.legend(fontsize=20, loc='upper right')

plt.tight_layout()

plt.show()